# Instance Segmentation with Mask R-CNN on Pascal VOC
This notebook demonstrates finetuning and evaluating a pretrained Mask R-CNN for Instance Segmentation on VOC dataset.

In [ ]:
import os

# Automatically detect the execution environment to configure the saving directory
if 'KAGGLE_URL_BASE' in os.environ:
    print("Kaggle Environment Detected")
    save_dir = '/kaggle/working/DL_Assignment_2/instance'
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("Google Colab Environment Detected")
        save_dir = '/content/drive/My Drive/DL_Assignment_2/instance'
    except ImportError:
        print("Local Environment Detected")
        save_dir = './DL_Assignment_2/instance'

os.makedirs(save_dir, exist_ok=True)
print(f"Results will be saved at: {save_dir}")

In [ ]:
!pip install -q torchmetrics

# Install and import libraries
import torch
import torchvision
import torchvision.transforms.functional as F
from torchvision.datasets import VOCSegmentation
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm

## 1. Custom Dataset Loading
For Instance Segmentation, we utilize the `SegmentationObject` masks from the VOC dataset. Each distinct pixel value corresponds to a unique object, enabling the extraction of individual bounding boxes and binary masks.

In [ ]:
class VOCInstanceDataset(torch.utils.data.Dataset):
    def __init__(self, root, year, image_set, download=False):
        self.voc = VOCSegmentation(root, year=year, image_set=image_set, download=download)
        
    def __getitem__(self, idx):
        img_path = self.voc.images[idx]
        mask_path = self.voc.masks[idx].replace('SegmentationClass', 'SegmentationObject') # Map standard masks to Object-level segmentation masks
        
        img = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)
        
        mask = np.array(mask)
        obj_ids = np.unique(mask)
        # Exclude margin and background artifacts
        obj_ids = obj_ids[(obj_ids != 0) & (obj_ids != 255)]
        masks = mask == obj_ids[:, None, None]
        num_objs = len(obj_ids)
        
        boxes = []
        valid_masks = []
        for i in range(num_objs):
            pos = np.where(masks[i])
            xmin = np.min(pos[1])
            xmax = np.max(pos[1])
            ymin = np.min(pos[0])
            ymax = np.max(pos[0])
            if xmax > xmin and ymax > ymin:
                boxes.append([xmin, ymin, xmax, ymax])
                valid_masks.append(masks[i])
        
        num_objs = len(boxes)
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        # Treat all instances as class 1 (Foreground)
        labels = torch.ones((num_objs,), dtype=torch.int64)
        masks_tensor = torch.as_tensor(np.array(valid_masks), dtype=torch.uint8)
        image_id = torch.tensor([idx])
        
        if num_objs == 0:
            boxes = torch.empty((0, 4), dtype=torch.float32)
            labels = torch.empty((0,), dtype=torch.int64)
            masks_tensor = torch.empty((0, mask.shape[0], mask.shape[1]), dtype=torch.uint8)
            
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]) if num_objs > 0 else torch.zeros((0,))
        iscrowd = torch.zeros((num_objs,), dtype=torch.int64)
        
        target = {
            "boxes": boxes,
            "labels": labels,
            "masks": masks_tensor,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd
        }

        img = F.to_tensor(img)
        return img, target

    def __len__(self):
        return len(self.voc)

def collate_fn(batch):
    return tuple(zip(*batch))

train_dataset = VOCInstanceDataset(root='./data', year='2012', image_set='train', download=True)
val_dataset = VOCInstanceDataset(root='./data', year='2012', image_set='val', download=True)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2, collate_fn=collate_fn)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

## 2. Model Initialization
Replace the pre-trained Mask R-CNN classifier to output 2 classes (Foreground and Background).

In [ ]:
def get_model_instance_segmentation(num_classes):
    model = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)
    
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)
    
    return model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_classes = 2 # Background and Foreground classes
model = get_model_instance_segmentation(num_classes)
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

## 3. Training & Evaluation Pipeline
Utilize the mean Average Precision (mAP) metric for bounding box evaluation and model checkpointing.

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

epochs = 20
train_losses = []
val_map = []
val_map_50 = []
val_map_75 = []
start_epoch = 0
checkpoint_path = os.path.join(save_dir, 'instance_checkpoint.pth')
best_model_path = os.path.join(save_dir, 'instance_best_maskrcnn_voc.pth')

best_map = 0.0

import os
if os.path.exists(checkpoint_path):
    print("Found checkpoint, loading to resume training...")
    checkpoint = torch.load(checkpoint_path, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    train_losses = checkpoint['train_losses']
    val_map = checkpoint['val_map']
    val_map_50 = checkpoint.get('val_map_50', [])
    val_map_75 = checkpoint.get('val_map_75', [])
    
    # Restore Early stopping states
    best_map = checkpoint.get('best_map', 0.0)
    print(f"Resuming training from epoch {start_epoch + 1}")

for epoch in range(start_epoch, epochs):
    model.train()
    running_loss = 0.0
    for images, targets in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]'):
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        valid_idx = [i for i, t in enumerate(targets) if len(t["boxes"]) > 0]
        if len(valid_idx) == 0:
            continue
            
        images = [images[i] for i in valid_idx]
        targets = [targets[i] for i in valid_idx]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        running_loss += losses.item()
        
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    
    # Eval
    model.eval()
    metric = MeanAveragePrecision(iou_type="bbox")
    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Val]'):
            images = list(image.to(device) for image in images)
            outputs = model(images)
            
            p_preds = []
            p_targets = []
            for i in range(len(outputs)):
                pred_dict = {'boxes': outputs[i]['boxes'].cpu(), 'scores': outputs[i]['scores'].cpu(), 'labels': outputs[i]['labels'].cpu()}
                target_dict = {'boxes': targets[i]['boxes'].cpu(), 'labels': targets[i]['labels'].cpu()}
                if len(target_dict['boxes']) > 0:
                    p_preds.append(pred_dict)
                    p_targets.append(target_dict)
            if p_preds:    
                metric.update(p_preds, p_targets)
                
    try:
        mAP_results = metric.compute()
        map_score = mAP_results['map'].item()
        map_50 = mAP_results['map_50'].item()
        map_75 = mAP_results['map_75'].item()
    except Exception as e:
        map_score = 0.0
        map_50 = 0.0
        map_75 = 0.0
    val_map.append(map_score)
    val_map_50.append(map_50)
    val_map_75.append(map_75)
    print(f"Epoch [{epoch+1}/{epochs}] Loss: {avg_loss:.4f}, mAP: {map_score:.4f}, mAP@0.5: {map_50:.4f}, mAP@0.75: {map_75:.4f}")

    # BEST MODEL SAVING LOGIC
    if map_score > best_map:
        best_map = map_score
        # Save optimal model
        torch.save(model.state_dict(), best_model_path)
        print(f"--> Optimal model saved with mAP: {best_map:.4f}")

    # SAVE TRAINING CHECKPOINT
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_losses': train_losses,
        'val_map': val_map,
        'val_map_50': val_map_50,
        'val_map_75': val_map_75,
        'best_map': best_map
    }, checkpoint_path)


torch.save(model.state_dict(), os.path.join(save_dir, 'instance_maskrcnn_voc_last.pth'))
print("Training completed! The optimal model weights are saved at:", best_model_path)

## 4. Visualization & Result Analysis

In [ ]:
# Plot Training Curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, epochs+1), train_losses, marker='o', color='purple')
plt.title('Training Total Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, epochs+1), val_map, marker='o', color='green', label='mAP@[0.5:0.95]')
plt.plot(range(1, epochs+1), val_map_50, marker='s', color='orange', label='mAP@0.5')
plt.plot(range(1, epochs+1), val_map_75, marker='^', color='red', label='mAP@0.75')
plt.title('Validation mAP (BBox)')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(save_dir, 'instance_training_curves.png'))
plt.show()

## 5. Final Model Evaluation

In [ ]:
# 1. Load optimal model weights
if os.path.exists(best_model_path):
    print(f"Loading best weights from: {best_model_path}")
    model.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=False))
else:
    print("Warning: Optimal checkpoint not found. Evaluating using current model state.")

model.to(device)
model.eval()

# 2. Initialize mAP metric calculator
from torchmetrics.detection.mean_ap import MeanAveragePrecision
final_metric = MeanAveragePrecision(iou_type="bbox")
final_metric.to(device)

# 3. Execute final validation epoch
print("Executing comprehensive evaluation on the validation dataset...")
with torch.no_grad():
    for images, targets in tqdm(val_loader, desc='Final Validation'):
        images = list(image.to(device) for image in images)
        outputs = model(images)
        
        p_preds = []
        p_targets = []
        for i in range(len(outputs)):
            pred_dict = {'boxes': outputs[i]['boxes'].cpu(), 'scores': outputs[i]['scores'].cpu(), 'labels': outputs[i]['labels'].cpu()}
            target_dict = {'boxes': targets[i]['boxes'].cpu(), 'labels': targets[i]['labels'].cpu()}
            if len(target_dict['boxes']) > 0:
                p_preds.append(pred_dict)
                p_targets.append(target_dict)
        if p_preds:    
            final_metric.update(p_preds, p_targets)

# 4. Generate Metric Report
final_results = final_metric.compute()

map_coco = final_results['map'].item()
map_50 = final_results['map_50'].item()
map_75 = final_results['map_75'].item()

print("\n" + "=" * 55)
print(f"   FINAL INSTANCE EVALUATION METRICS")
print("=" * 55)
print(f"[+] mAP ([0.5:0.95]) : {map_coco:.4f}")
print(f"[+] mAP (@0.5)        : {map_50:.4f}")
print(f"[+] mAP (@0.75)       : {map_75:.4f}")
print("-" * 55)
print("--> Metrics computation finished.")


In [ ]:
import colorsys

def random_colors(N, bright=True):
    brightness = 1.0 if bright else 0.7
    hsv = [(i / N, 1, brightness) for i in range(N)]
    colors = list(map(lambda c: colorsys.hsv_to_rgb(*c), hsv))
    return list(map(lambda c: (int(c[0]*255), int(c[1]*255), int(c[2]*255)), colors))

print("Executing automated search for complex overlapping object scenarios...")
target_indices = []
for i in range(len(val_dataset)):
    _, target = val_dataset[i]
    labels = target['labels']
    if len(labels) >= 3 and len(torch.unique(labels)) < len(labels):
        target_indices.append(i)
        if len(target_indices) >= 2:
            break

if len(target_indices) == 0:
    target_indices = [0, 1]

images = [val_dataset[i][0] for i in target_indices]

model.eval()
images_gpu = list(img.to(device) for img in images)
with torch.no_grad():
    predictions = model(images_gpu)

for i in range(len(images)):
    img = images[i].permute(1, 2, 0).numpy().copy()
    
    pred = predictions[i]
    boxes = pred['boxes'].cpu().numpy()
    scores = pred['scores'].cpu().numpy()
    masks = pred['masks'].cpu().numpy()
    
    threshold = 0.5
    valid_idxs = np.where(scores > threshold)[0]
    
    colors = random_colors(max(1, len(valid_idxs)))
    
    fig, ax = plt.subplots(1, figsize=(10, 8))
    ax.imshow(img)
    
    for idx, obj_idx in enumerate(valid_idxs):
        c_idx = idx % len(colors)
        box = boxes[obj_idx]
        mask = masks[obj_idx, 0] > 0.5
        
        rect = patches.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1], 
                                 linewidth=2, edgecolor=np.array(colors[c_idx])/255., facecolor='none')
        ax.add_patch(rect)
        
        colored_mask = np.zeros((*mask.shape, 3))
        for c in range(3): colored_mask[:, :, c] = colors[c_idx][c]/255.
        ax.imshow(np.dstack([colored_mask[:, :, 0], colored_mask[:, :, 1], colored_mask[:, :, 2], mask * 0.5]))
        ax.text(box[0], box[1] - 5, f'Obj: {scores[obj_idx]:.2f}', color='white',
               bbox=dict(facecolor=np.array(colors[c_idx])/255., alpha=0.5))
               
    ax.axis('off')
    plt.title("Mask R-CNN Predicted Mask\nPhân tách đa thực thể rõ nét")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'instance_predictions_{i}.png'))
    plt.show()

## 6. Resource Efficiency Analysis

In [ ]:
import time

# 1. Parameter Computation
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"=== RESOURCE REPORT (Mask R-CNN) ===")
print(f"- Total Parameters: {total_params / 1e6:.2f} M")
print(f"- Trainable Parameters: {trainable_params / 1e6:.2f} M")

# 2. Inference Speed Computation (FPS)
model.eval()
print("\nEvaluating Inference Speed (FPS) across 50 sample images...")

# Hardware Warm-up execution
dummy_img = [torch.randn(3, 256, 256).to(device)]
with torch.no_grad():
    for _ in range(5):
        model(dummy_img)

start_time = time.time()
num_test = min(50, len(val_dataset))
with torch.no_grad():
    for i in range(num_test):
        img_tensor = val_dataset[i][0].to(device)
        _ = model([img_tensor])

if torch.cuda.is_available():
    torch.cuda.synchronize()
end_time = time.time()

total_time = end_time - start_time
fps = num_test / total_time
print(f"\n=== PERFORMANCE EVALUATION REPORT ===")
print(f"- Total inference time for {num_test} images: {total_time:.4f} giây")
print(f"- Processed Frames Per Second (FPS): {fps:.2f} FPS")
print(f"- Average latency per image: {(total_time/num_test)*1000:.2f} ms")
